# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
This dataset provides mass spectrometry analysis results from human mast cell samples, structured by a Croissant schema.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 mass spectrometry dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Keep as object

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, their IDs, and associated fields or columns. Each entity is referenced by its Croissant `@id`.

In [ ]:
# List all record sets in the dataset
print("Record sets in dataset (by @id):")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- {record_set.id}")
    record_sets.append(record_set.id)
    # Print basic information about the record set
    print(f"  Name: {getattr(record_set, 'name', '')}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    # List columns/fields of this record set
    field_ids = []
    for field in getattr(record_set, 'fields', []):
        print(f"    Field @id: {field.id}, name: {getattr(field, 'name', '')}, dataType: {getattr(field, 'data_type', '')}")
        field_ids.append(field.id)
    print("")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use record set and field `@id`s.

*Note: If the dataset includes multiple record sets, they will all be shown. Typically, you can focus on the main tabular one for analysis.*

In [ ]:
dataframes = {}

# Preview all available record sets again (by @id):
print(f"Available record set ids: {record_sets}\n")

# For this dataset, let's focus on the main record set (the first one)
if record_sets:
    main_record_set = record_sets[0]
else:
    raise ValueError("No record sets found in the dataset.")
print(f"Extracting records from record set: {main_record_set}")

records = list(dataset.records(record_set=main_record_set))
df = pd.DataFrame(records)
dataframes[main_record_set] = df

print(f"Columns from record set {main_record_set}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We now apply basic exploration and analysis steps: filtering, normalization, and grouping by attributes. All fields are referenced by their Croissant `@id`s.

In [ ]:
# Show all field @id -> column mappings to guide EDA selection
record_set_obj = None
for rs in metadata.record_sets:
    if rs.id == main_record_set:
        record_set_obj = rs
        break

field_ids = [field.id for field in getattr(record_set_obj, 'fields', [])]
print("Fields (as @id):", field_ids)

# Choose a numeric field for demonstration. Replace with a real field name if necessary.
# Example: select the first float/int field available.
numeric_field_id = None
for field in getattr(record_set_obj, 'fields', []):
    if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
        numeric_field_id = field.id
        break
if numeric_field_id is None:
    # Fall back to first field
    numeric_field_id = field_ids[0]

print(f"Using {numeric_field_id} as numeric field for analysis.")

# Filter the dataframe for numeric_field_id > threshold
threshold = 10
if numeric_field_id in df.columns:
    # Attempt to convert to numeric (if necessary)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field if available
    group_field_id = None
    for field in getattr(record_set_obj, 'fields', []):
        if getattr(field, 'data_type', '').lower() == 'text':
            group_field_id = field.id
            break
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print(f"Field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize the distribution of the selected numeric field using a histogram. We can also plot the normalized values or group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'filtered_df' in locals() and f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.kdeplot(filtered_df[f"{numeric_field_id}_normalized"], fill=True)
        plt.title(f"Normalized {numeric_field_id} (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.ylabel("Density")
        plt.show()
else:
    print(f"Numeric field {numeric_field_id} not available for plotting.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using its Croissant schema, inspected the data catalog by their `@id`s, and demonstrated basic filtering, normalization, grouping, and visualization of a numeric feature.

**Key findings:**
- The dataset provides structured protein analysis records with rich metadata and ready-to-analyze records.
- Using `mlcroissant`, record sets and fields may be programmatically discovered and referenced by their Croissant `@id`.
- The notebook style here is fully general: you can modify the example EDA or visualization code to analyze other fields or record sets, always referencing their `@id` for best reproducibility.